# Score New Claims

Inference-only notebook. Loads the model trained in `Regression3.ipynb` (via `approved_amount_model.joblib`) and answers three things for a new claim, per `architecture_pivot_instructions.txt`:

1. **`score_new_claim()`** -- predicts `approved_amount` from the claim's characteristics (never its dollar amounts), compares it to the shop's actual `invoice_amount`, flags it if the gap exceeds `threshold_pct`.
2. **`find_similar_historical_claims()`** -- looks up the k most similar past claims (by make/model/damage/region/parts-action profile) to whatever gets flagged.
3. **`explain_deviation()`** -- turns that comparison into an actual structured answer to "why does this cost more": a labor-rate delta, a parts-action delta, and which specific parts this claim replaced that similar claims usually just repaired.

**All three require `historical_df`, not just the trained model** -- the shop-average lookup and the similarity/explanation comparisons must reflect current historical data at call time, not a value frozen when the model was trained. `historical_df` is built fresh in this notebook from `Claim_main.csv` + `repair_parts_action_summary.csv` + `repair_line_items_with_action.csv` (for labor rates) every time this notebook runs.

**Requires `Regression3.ipynb` to have been run first**, so `approved_amount_model.joblib` exists in this folder.

Two ways to score new claims:
- **CSV mode** -- `new_claims_template.csv`, one row per claim, batch-scored (predicted amount + flag only).
- **Manual mode** -- edit the `new_claim` dict directly, get the full score + similar claims + explanation for one claim.

In [1]:
import pandas as pd
import numpy as np
import joblib

## Load the saved model

In [2]:
artifact = joblib.load("approved_amount_model.joblib")

model = artifact["model"]
NUMERIC_FEATURES = artifact["numeric_features"]
CATEGORICAL_FEATURES = artifact["categorical_features"]
SEVERITY_ORDER = artifact["severity_order"]
TARGET = artifact["target"]
DEFAULT_THRESHOLD_PCT = artifact["default_threshold_pct"]

print(f"Loaded {artifact['model_name']} model, trained to predict {TARGET}")
print(f"Numeric features: {NUMERIC_FEATURES}")
print(f"Categorical features: {CATEGORICAL_FEATURES}")

Loaded HistGradientBoosting model, trained to predict approved_amount
Numeric features: ['vehicle_age', 'liability_percentage', 'repair_duration_days', 'vehicle_claim_count', 'damage_severity_ordinal', 'initial_damage_assessment_ordinal', 'airbags_deployed', 'is_supplemental', 'has_catastrophe_code', 'reserve_amount', 'is_repeat_vehicle', 'shop_avg_approved_amount', 'replaced_cost_total', 'repaired_cost_total', 'reused_cost_total', 'num_parts_replaced', 'num_parts_repaired', 'num_parts_reused', 'pct_cost_replaced']
Categorical features: ['make', 'model', 'claim_severity', 'incident_state', 'loss_cause', 'point_of_impact', 'weather_conditions', 'loss_month', 'shop_city', 'shop_state']


## Feature engineering

Same transformations `Regression3.ipynb` applies, minus the shop-average lookup (that now happens inside `score_new_claim`, against live `historical_df`, not here). Returns every raw + derived column, not just the model's feature list -- `find_similar_historical_claims`/`explain_deviation` need raw fields (`make`, `damage_severity` as a string, the named part slots) that aren't part of `X`.

In [3]:
def engineer_features(raw_df):
    df = raw_df.copy()

    df["vehicle_age"] = pd.Timestamp.now().year - df["vehicle_year"]
    df["damage_severity_ordinal"] = df["damage_severity"].map(SEVERITY_ORDER)
    df["initial_damage_assessment_ordinal"] = df["initial_damage_assessment"].map(SEVERITY_ORDER)
    df["airbags_deployed"] = df["airbags_deployed"].astype(int)
    df["is_supplemental"] = (df["cost_category"] == "Supplemental").astype(int)
    df["has_catastrophe_code"] = df["catastrophe_code"].notna().astype(int)
    df["is_repeat_vehicle"] = df["is_repeat_vehicle"].astype(int)

    df["loss_date"] = pd.to_datetime(df["loss_date"])
    df["loss_month"] = df["loss_date"].dt.month

    df["replaced_cost_total"] = df["replaced_cost_total"].fillna(0)
    df["repaired_cost_total"] = df["repaired_cost_total"].fillna(0)
    df["reused_cost_total"] = df["reused_cost_total"].fillna(0)
    df["num_parts_replaced"] = df["num_parts_replaced"].fillna(0)
    df["num_parts_repaired"] = df["num_parts_repaired"].fillna(0)
    df["num_parts_reused"] = df["num_parts_reused"].fillna(0)
    total_parts_cost = df["replaced_cost_total"] + df["repaired_cost_total"] + df["reused_cost_total"]
    df["pct_cost_replaced"] = (df["replaced_cost_total"] / total_parts_cost).fillna(0)

    for i in range(1, 6):
        df[f"replaced_part_{i}"] = df[f"replaced_part_{i}"].fillna("None")
        df[f"replaced_part_{i}_cost"] = df[f"replaced_part_{i}_cost"].fillna(0)
        df[f"repaired_part_{i}"] = df[f"repaired_part_{i}"].fillna("None")
        df[f"repaired_part_{i}_cost"] = df[f"repaired_part_{i}_cost"].fillna(0)

    return df

## Build `historical_df`

Loaded fresh every run -- this is deliberate, not a shortcut: the shop-average lookup and the similar-claims comparison must reflect whatever historical data exists right now, not a snapshot frozen at training time.

Adds one thing `Regression3.ipynb` doesn't need: `avg_labor_rate` per `repair_id`, aggregated from `repair_line_items_with_action.csv` (mean `labor_rate` across that repair's `Labor`/`Paint Labor` line items -- `labor_rate` isn't populated for other line item types). This is required for `explain_deviation`'s regional labor-rate comparison. Repairs with no labor line items fall back to the overall mean.

In [4]:
claims = pd.read_csv("Claim_main.csv")
parts_action = pd.read_csv("repair_parts_action_summary.csv")
line_items = pd.read_csv("repair_line_items_with_action.csv")

merged = claims.merge(parts_action.drop(columns=["claim_id", "vehicle_id"]), on="repair_id", how="inner", validate="one_to_one")

avg_labor_rate = (line_items[line_items["line_item_type"].isin(["Labor", "Paint Labor"])]
    .groupby("repair_id")["labor_rate"].mean().rename("avg_labor_rate"))
merged = merged.merge(avg_labor_rate, on="repair_id", how="left")
merged["avg_labor_rate"] = merged["avg_labor_rate"].fillna(merged["avg_labor_rate"].mean())

historical_df = engineer_features(merged)
print(f"historical_df: {len(historical_df)} rows")
historical_df[["repair_id", "repair_shop_contact_id", "avg_labor_rate", "pct_cost_replaced"]].head()

historical_df: 10241 rows


,repair_id,repair_shop_contact_id,avg_labor_rate,pct_cost_replaced
0,1,209,342.183333,0.561086
1,2,12,118.465000,0.000000
2,3,88,290.460000,0.000000
3,4,164,225.580000,1.000000
4,5,106,780.350000,0.000000


## `score_new_claim()`

`historical_df` is a required parameter -- the shop average is looked up from it at call time, not baked into the saved model.

In [5]:
def score_new_claim(new_claim_features, trained_model, historical_df, invoice_amount, threshold_pct=DEFAULT_THRESHOLD_PCT):
    shop_avg = historical_df.loc[
        historical_df["repair_shop_contact_id"] == new_claim_features["repair_shop_contact_id"], TARGET
    ].mean()
    if pd.isna(shop_avg):
        shop_avg = historical_df[TARGET].mean()

    feature_row = dict(new_claim_features)
    feature_row[f"shop_avg_{TARGET}"] = shop_avg
    X_new = pd.DataFrame([feature_row])[NUMERIC_FEATURES + CATEGORICAL_FEATURES]

    predicted = trained_model.predict(X_new)[0]
    deviation_pct = (invoice_amount - predicted) / predicted
    return {
        "predicted_approved_amount": predicted,
        "invoice_amount": invoice_amount,
        "deviation_pct": deviation_pct,
        "flagged": deviation_pct > threshold_pct,  # matches architecture_pivot_instructions.txt literally -- NOT abs(), only catches overbilling
    }

## `find_similar_historical_claims()`

If this vehicle (`vehicle_id`) already has repair history in `historical_df`, compare against *that vehicle's own past claims first* -- same car, controls for everything except what actually changed over time (labor rates, what got replaced vs repaired). Only falls back to ranking `historical_df` by similarity (make/model/damage_severity/incident_state exact match, then distance on vehicle_year + parts-action profile) if this vehicle has no history of its own. Returns the k closest rows including their actual dollar amounts, named part slots, and `avg_labor_rate` -- everything `explain_deviation` needs -- plus `_match_source` so it's clear which comparison was actually used.

In [6]:
SIMILARITY_CATEGORICAL_COLS = ["make", "model", "damage_severity", "incident_state"]
SIMILARITY_NUMERIC_COLS = ["vehicle_year", "num_parts_replaced", "num_parts_repaired", "num_parts_reused", "pct_cost_replaced"]

def find_similar_historical_claims(new_claim_features, historical_df, k=10):
    same_vehicle = historical_df[historical_df["vehicle_id"] == new_claim_features.get("vehicle_id")]
    if len(same_vehicle) > 0:
        candidates = same_vehicle.copy()
        match_source = "same_vehicle"
    else:
        candidates = historical_df.copy()
        match_source = "similar_claims"

    match_score = pd.Series(0, index=candidates.index)
    for col in SIMILARITY_CATEGORICAL_COLS:
        match_score += (candidates[col] == new_claim_features[col]).astype(int)

    numeric_dist = pd.Series(0.0, index=candidates.index)
    for col in SIMILARITY_NUMERIC_COLS:
        col_std = candidates[col].std() or 1.0
        numeric_dist += ((candidates[col] - new_claim_features[col]) / col_std).abs()

    candidates["_match_score"] = match_score
    candidates["_numeric_dist"] = numeric_dist
    candidates["_match_source"] = match_source
    candidates = candidates.sort_values(["_match_score", "_numeric_dist"], ascending=[False, True])

    return_cols = (["repair_id", "claim_id", "vehicle_id", "make", "model", "vehicle_year", "damage_severity", "incident_state",
        "invoice_amount", "approved_amount", "avg_labor_rate",
        "num_parts_replaced", "num_parts_repaired", "num_parts_reused", "pct_cost_replaced"]
        + [f"replaced_part_{i}" for i in range(1, 6)] + [f"repaired_part_{i}" for i in range(1, 6)]
        + ["_match_score", "_numeric_dist", "_match_source"])

    return candidates[return_cols].head(k)

## `explain_deviation()`

The actual "why" answer -- not just a table of similar claims, a computed breakdown: how this claim's labor rate and parts-action mix compare to similar claims' averages, plus which specific parts it replaced that similar claims typically just repaired.

In [7]:
def explain_deviation(new_claim_features, similar_claims):
    labor_rate_new = new_claim_features["avg_labor_rate"]
    labor_rate_similar = similar_claims["avg_labor_rate"].mean()
    labor_rate_delta = labor_rate_new - labor_rate_similar
    labor_rate_delta_pct = labor_rate_delta / labor_rate_similar if labor_rate_similar else None

    pct_replaced_new = new_claim_features["pct_cost_replaced"]
    pct_replaced_similar = similar_claims["pct_cost_replaced"].mean()
    pct_cost_replaced_delta = pct_replaced_new - pct_replaced_similar

    new_replaced_parts = {new_claim_features.get(f"replaced_part_{i}") for i in range(1, 6)} - {None, "None"}
    similar_repaired_parts = set()
    for i in range(1, 6):
        similar_repaired_parts |= set(similar_claims[f"repaired_part_{i}"].dropna().unique()) - {"None"}
    parts_replaced_not_typical = sorted(new_replaced_parts & similar_repaired_parts)

    return {
        "labor_rate_delta": labor_rate_delta,
        "labor_rate_delta_pct": labor_rate_delta_pct,
        "pct_cost_replaced_delta": pct_cost_replaced_delta,
        "parts_replaced_not_typical": parts_replaced_not_typical,
    }

## Mode B: score one claim typed by hand, with the full explanation

Edit the values below, then run this cell -- it scores the claim, finds similar historical claims, and explains the deviation.

In [13]:
new_claim_raw = pd.DataFrame([{
    "claim_id": "manual-1",
    "vehicle_id": 1358,                        # set to an existing vehicle_id to demo the same-vehicle comparison; use a new/unused id for a brand-new vehicle
    "repair_shop_contact_id": 209,
    "vehicle_year": 2015,
    "liability_percentage": 100,
    "repair_duration_days": 10,
    "vehicle_claim_count": 1,
    "damage_severity": "Moderate",             # Minor / Moderate / Severe / Total Loss
    "initial_damage_assessment": "Moderate",   # same scale
    "airbags_deployed": 0,                     # 0 or 1
    "cost_category": "Standard",               # "Supplemental" or anything else
    "catastrophe_code": None,                  # a code string, or None
    "is_repeat_vehicle": 0,                    # 0 or 1
    "num_parts_replaced": 1,
    "num_parts_repaired": 0,
    "num_parts_reused": 0,
    "replaced_cost_total": 900.00,
    "repaired_cost_total": 0.00,
    "reused_cost_total": 0.00,
    "replaced_part_1": "Headlight Assembly", "replaced_part_1_cost": 900.00,
    "replaced_part_2": None, "replaced_part_2_cost": 0.00,
    "replaced_part_3": None, "replaced_part_3_cost": 0.00,
    "replaced_part_4": None, "replaced_part_4_cost": 0.00,
    "replaced_part_5": None, "replaced_part_5_cost": 0.00,
    "repaired_part_1": None, "repaired_part_1_cost": 0.00,
    "repaired_part_2": None, "repaired_part_2_cost": 0.00,
    "repaired_part_3": None, "repaired_part_3_cost": 0.00,
    "repaired_part_4": None, "repaired_part_4_cost": 0.00,
    "repaired_part_5": None, "repaired_part_5_cost": 0.00,
    "avg_labor_rate": 340.00,                  # this claim's quoted/estimated labor rate
    "make": "Mercedes-Benz",
    "model": "C-Class",
    "claim_severity": "Moderate",
    "incident_state": "CA",
    "loss_cause": "Collision",
    "point_of_impact": "Front",
    "weather_conditions": "Clear",
    "shop_city": "Los Angeles",
    "shop_state": "CA",
    "loss_date": "2026-06-01",
    "reserve_amount": 1800.00,
    "invoice_amount": 2400.00,                 # what the shop actually billed -- comparison only, not a model input
}])

new_claim_features = engineer_features(new_claim_raw).iloc[0].to_dict()

score = score_new_claim(new_claim_features, model, historical_df, new_claim_features["invoice_amount"])
print("SCORE:", score)

similar = find_similar_historical_claims(new_claim_features, historical_df, k=10)
print(f"\\n{len(similar)} comparison claims found (source: {similar['_match_source'].iloc[0]})")
display(similar)

explanation = explain_deviation(new_claim_features, similar)
print("\\nEXPLANATION:", explanation)

SCORE: {'predicted_approved_amount': 3323.983692970748, 'invoice_amount': 2400.0, 'deviation_pct': -0.277974797206347, 'flagged': False}
\n10 comparison claims found (source: same_vehicle)


,repair_id,claim_id,vehicle_id,make,model,vehicle_year,damage_severity,incident_state,invoice_amount,approved_amount,...,replaced_part_4,replaced_part_5,repaired_part_1,repaired_part_2,repaired_part_3,repaired_part_4,repaired_part_5,_match_score,_numeric_dist,_match_source
5715,5716,4118,1358,Mercedes-Benz,C-Class,2015,Moderate,CA,4266.29,4304.61,...,None,None,None,None,None,None,None,4,0.000000,same_vehicle
5717,5718,4118,1358,Mercedes-Benz,C-Class,2015,Moderate,CA,3982.16,3732.31,...,None,None,None,None,None,None,None,4,1.154701,same_vehicle
5716,5717,4118,1358,Mercedes-Benz,C-Class,2015,Moderate,CA,3889.03,3713.89,...,None,None,Rear Bumper Cover,Tail Light Assembly,None,None,None,4,6.126392,same_vehicle
1324,1325,954,1358,Mercedes-Benz,C-Class,2015,Moderate,NJ,4558.64,4720.28,...,None,None,None,None,None,None,None,3,0.000000,same_vehicle
1326,1327,954,1358,Mercedes-Benz,C-Class,2015,Moderate,NJ,4916.60,4611.46,...,None,None,None,None,None,None,None,3,1.154701,same_vehicle
6329,6330,4548,1358,Mercedes-Benz,C-Class,2015,Moderate,TX,3673.14,3821.93,...,None,None,Rear Bumper Cover,None,None,None,None,3,1.998012,same_vehicle
1325,1326,954,1358,Mercedes-Benz,C-Class,2015,Moderate,NJ,4950.02,4703.29,...,None,None,Rear Bumper Cover,None,None,None,None,3,4.630635,same_vehicle
6328,6329,4548,1358,Mercedes-Benz,C-Class,2015,Moderate,TX,3840.28,3890.00,...,None,None,Quarter Panel,None,None,None,None,3,4.630635,same_vehicle
8697,8698,6245,1358,Mercedes-Benz,C-Class,2015,Moderate,FL,5100.12,4977.95,...,None,None,Grille,None,None,None,None,3,4.630635,same_vehicle
8698,8699,6245,1358,Mercedes-Benz,C-Class,2015,Moderate,FL,5203.54,4919.11,...,None,None,None,None,None,None,None,3,5.703925,same_vehicle


\nEXPLANATION: {'labor_rate_delta': -14.506666666666774, 'labor_rate_delta_pct': -0.04092071611253226, 'pct_cost_replaced_delta': 0.5253641227774464, 'parts_replaced_not_typical': []}


## Mode A: score a CSV of claims

Batch mode -- prediction + flag only for every row (no similar-claims/explanation detail, that's manual mode below). File path is a variable (`CSV_PATH`) so you can point it at any CSV with the right columns -- see `new_claims_template.csv` for the expected columns and example rows.

In [12]:
CSV_PATH = "gemini-code.csv"  # change to your own file

raw_claims = pd.read_csv(CSV_PATH)
random_num = np.random.randint(0, len(raw_claims))
engineered = engineer_features(raw_claims.iloc[[random_num]])

rows = []
for _, row in engineered.iterrows():
    result = score_new_claim(row.to_dict(), model, historical_df, row["invoice_amount"])
    rows.append({"claim_id": row["claim_id"], **result})

pd.DataFrame(rows)

,claim_id,predicted_approved_amount,invoice_amount,deviation_pct,flagged
0,CLM-10293,1390.101077,2728.91,0.963102,True
